# NumPy for Machine Learning: Memory Layout, Strides, and Vectorization
Accompanies **Appendix A: NumPy for Machine Learning**.

Standard Python objects (`PyObject`) incur 28 bytes of heap overhead per number and are accessed via pointers, creating CPU cache misses. 
NumPy's `ndarray` stores homogeneous binary buffers in contiguous hardware memory, indexed by shape and byte strides.

In this notebook, we examine memory strides, benchmark SIMD vectorization vs. Python loops, and explore broadcasting mechanics.

In [ ]:
import time
import numpy as np

# A 2x3 matrix of 64-bit floats (8 bytes per entry)
A = np.array([[1.0, 2.0, 3.0], 
              [4.0, 5.0, 6.0]], dtype=np.float64)

print(f"Shape:   {A.shape}")
print(f"Strides: {A.strides} -> (24 bytes to jump rows, 8 bytes to jump columns)")
print(f"Transposed Strides: {A.T.strides} -> Zero memory copied; stride swap view!")

### Benchmarking Vectorization vs. Python Loops
NumPy delegates operations to precompiled C routines leveraging CPU SIMD registers (AVX/NEON) to process multiple floats per cycle.

In [ ]:
n = 1_000_000
x = np.random.randn(n)
y = np.random.randn(n)

# Python loop
start = time.time()
diff_sum = sum((a - b) ** 2 for a, b in zip(x, y)) ** 0.5
time_loop = time.time() - start

# Vectorized NumPy
start = time.time()
diff_np = np.sqrt(np.sum((x - y) ** 2))
time_np = time.time() - start

print(f"Python loop: {time_loop:.4f} s")
print(f"Vectorized:  {time_np:.4f} s")
print(f"Speedup:     {time_loop / time_np:.1f}x faster")

### Broadcasting Alignment Rules
Two dimensions are compatible when they are equal or one of them is 1.
NumPy aligns shapes from right to left (trailing dimensions).

In [ ]:
# Feature centering: Subtracting column means from a 500x5 matrix
X = np.random.randn(500, 5) * 8 + 40
mu = np.mean(X, axis=0)  # Shape: (5,)

# X:  (500, 5)
# mu:      (5,) -> Virtual alignment: (1, 5)
X_centered = X - mu

assert np.allclose(np.mean(X_centered, axis=0), 0)
print("Column means centered to zero without explicit loops or copies.")